In [208]:
### Interpretation

# Houses with higher overall quality generally have higher sale prices, indicating a strong positive relationship.

In [209]:
X2 = df.drop("SalePrice", axis=1)

y2 = df["SalePrice"].astype(float)

In [210]:
numeric_features = X2.select_dtypes(
    include=np.number
).columns

categorical_features = X2.select_dtypes(
    exclude=np.number
).columns

print("Numerical Features :", len(numeric_features))
print("Categorical Features:", len(categorical_features))

Numerical Features : 37
Categorical Features: 43


In [212]:
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

In [213]:
preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

In [214]:
X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2,
    y2,
    test_size=0.2,
    random_state=42
)

print("Training Set :", X2_train.shape)
print("Testing Set  :", X2_test.shape)

Training Set : (1168, 80)
Testing Set  : (292, 80)


In [215]:
def evaluate_pipeline(model_name, model):

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipeline.fit(X2_train, y2_train)

    pred = pipeline.predict(X2_test)

    mae = mean_absolute_error(y2_test, pred)
    mse = mean_squared_error(y2_test, pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y2_test, pred)

    n = len(y2_test)
    p = X2_train.shape[1]

    adjusted_r2 = 1 - (
        (1 - r2) * (n - 1) / (n - p - 1)
    )

    return [
        model_name,
        mae,
        mse,
        rmse,
        r2,
        adjusted_r2
    ]

In [216]:
dummy = evaluate_pipeline(
    "Dummy Regressor",
    DummyRegressor(strategy="mean")
)

dummy

['Dummy Regressor',
 62575.926451960964,
 7677095207.783831,
 np.float64(87619.03450611533),
 -0.0008824918802490256,
 -0.3803640053893482]

In [217]:
simple_feature = "OverallQual"

X_simple = df[[simple_feature]]
y_simple = df["SalePrice"].astype(float)

X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X_simple,
    y_simple,
    test_size=0.2,
    random_state=42
)

simple_model = LinearRegression()

simple_model.fit(X_train_s, y_train_s)

pred = simple_model.predict(X_test_s)

mae = mean_absolute_error(y_test_s, pred)
mse = mean_squared_error(y_test_s, pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test_s, pred)

n = len(y_test_s)
p = 1

adjusted_r2 = 1 - (
    (1-r2)*(n-1)/(n-p-1)
)

simple = [
    "Simple Linear Regression",
    mae,
    mse,
    rmse,
    r2,
    adjusted_r2
]

simple

['Simple Linear Regression',
 33343.24208697128,
 2681026163.509091,
 np.float64(51778.62651238531),
 0.6504677778896862,
 0.6492624943651679]

In [218]:
multiple = evaluate_pipeline(
    "Multiple Linear Regression",
    LinearRegression()
)

multiple

['Multiple Linear Regression',
 18360.441101160442,
 867102416.2566457,
 np.float64(29446.60279653063),
 0.8869536454076533,
 0.8440924683110289]

In [219]:
ridge = evaluate_pipeline(
    "Ridge Regression",
    Ridge(alpha=1.0)
)

ridge

['Ridge Regression',
 19092.326539987193,
 887158979.4826419,
 np.float64(29785.214108390122),
 0.884338820081554,
 0.8404862400176881]

In [220]:
lasso = evaluate_pipeline(
    "Lasso Regression",
    Lasso(alpha=0.001)
)

lasso

['Lasso Regression',
 18074.327547251123,
 800232058.8918074,
 np.float64(28288.373210416456),
 0.8956717045303775,
 0.8561159526935538]

In [221]:
decision_tree = evaluate_pipeline(
    "Decision Tree Regression",
    DecisionTreeRegressor(random_state=42)
)

decision_tree

['Decision Tree Regression',
 26678.383561643837,
 1692859696.890411,
 np.float64(41144.37624864923),
 0.7792975616468313,
 0.6956189120342555]

In [222]:
random_forest = evaluate_pipeline(
    "Random Forest Regression",
    RandomForestRegressor(random_state=42)
)

random_forest

['Random Forest Regression',
 17590.05691780822,
 818470191.2887418,
 np.float64(28608.918037715823),
 0.8932939526715523,
 0.852836683542283]

In [223]:
results_ames = pd.DataFrame([
    dummy,
    simple,
    multiple,
    ridge,
    lasso,
    decision_tree,
    random_forest
],
columns=[
    "Model",
    "MAE",
    "MSE",
    "RMSE",
    "R2 Score",
    "Adjusted R2"
])

results_ames

,Model,MAE,MSE,RMSE,R2 Score,Adjusted R2
0,Dummy Regressor,62575.926452,7.677095e+09,87619.034506,-0.000882,-0.380364
1,Simple Linear Regression,33343.242087,2.681026e+09,51778.626512,0.650468,0.649262
2,Multiple Linear Regression,18360.441101,8.671024e+08,29446.602797,0.886954,0.844092
3,Ridge Regression,19092.326540,8.871590e+08,29785.214108,0.884339,0.840486
4,Lasso Regression,18074.327547,8.002321e+08,28288.373210,0.895672,0.856116
5,Decision Tree Regression,26678.383562,1.692860e+09,41144.376249,0.779298,0.695619
6,Random Forest Regression,17590.056918,8.184702e+08,28608.918038,0.893294,0.852837


In [224]:
results_ames = results_ames.round({
    "MAE":2,
    "MSE":2,
    "RMSE":2,
    "R2 Score":4,
    "Adjusted R2":4
})

results_ames

,Model,MAE,MSE,RMSE,R2 Score,Adjusted R2
0,Dummy Regressor,62575.93,7.677095e+09,87619.03,-0.0009,-0.3804
1,Simple Linear Regression,33343.24,2.681026e+09,51778.63,0.6505,0.6493
2,Multiple Linear Regression,18360.44,8.671024e+08,29446.60,0.8870,0.8441
3,Ridge Regression,19092.33,8.871590e+08,29785.21,0.8843,0.8405
4,Lasso Regression,18074.33,8.002321e+08,28288.37,0.8957,0.8561
5,Decision Tree Regression,26678.38,1.692860e+09,41144.38,0.7793,0.6956
6,Random Forest Regression,17590.06,8.184702e+08,28608.92,0.8933,0.8528


In [225]:
results_ames = results_ames.sort_values(
    by="R2 Score",
    ascending=False
).reset_index(drop=True)

results_ames

,Model,MAE,MSE,RMSE,R2 Score,Adjusted R2
0,Lasso Regression,18074.33,8.002321e+08,28288.37,0.8957,0.8561
1,Random Forest Regression,17590.06,8.184702e+08,28608.92,0.8933,0.8528
2,Multiple Linear Regression,18360.44,8.671024e+08,29446.60,0.8870,0.8441
3,Ridge Regression,19092.33,8.871590e+08,29785.21,0.8843,0.8405
4,Decision Tree Regression,26678.38,1.692860e+09,41144.38,0.7793,0.6956
5,Simple Linear Regression,33343.24,2.681026e+09,51778.63,0.6505,0.6493
6,Dummy Regressor,62575.93,7.677095e+09,87619.03,-0.0009,-0.3804
